# MP13 — Gradient Descent from Scratch

Expanded executable lesson. Prerequisites: preceding mathematics lessons. Derive the calculation, check a hand example, reproduce the controlled experiment and interpret its limits. Learner mastery remains unassessed.

## Setup
Install the repository development dependencies. `python scripts/training_experiments.py` regenerates the experiment using existing checksum-pinned NAIP scenes; the notebook reads its committed record without downloading data. Arrays use sample-by-feature shape and float64.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(root/'src'))
from mp13_tools import linear_loss,initialize,mlp_loss,finite_difference_error
record=json.loads((root/'reports/mp13-mp15-training.json').read_text())


## Mathematics and worked example
A derivative describes the local rate of change of a scalar function. For f(w)=w², f'(w)=2w. A gradient stacks partial derivatives for several parameters. To reduce a differentiable loss, gradient descent moves opposite its gradient: theta_next=theta-eta*gradient, where eta>0 is the learning rate. A step can overshoot; a decreasing direction is only a local statement.

For N scalar samples, prediction p_i=w*x_i+b and loss L=(1/N)sum_i(p_i-y_i)². The chain rule gives dL/dw=(2/N)sum_i(p_i-y_i)x_i and dL/db=(2/N)sum_i(p_i-y_i). Our general code uses X of shape N×D, W of shape D×O and b of shape O, returning predictions N×O. Mean squared loss averages all N*O entries, so its gradient normalization uses that count.

Worked example: x=[1,2], y=[3,5], w=b=0. Residuals are [-3,-5], L=17, dL/dw=-13, dL/db=-8. A simultaneous eta=.1 update gives w=1.3,b=.8, predictions [2.1,3.4] and L=1.685. Updating b before computing the w derivative would implement a different algorithm.

Central finite differences approximate a derivative by (L(theta+epsilon)-L(theta-epsilon))/(2epsilon). Too-large epsilon introduces truncation error; too-small epsilon magnifies roundoff. Compare several tolerances in float64 rather than demanding bitwise equality.

In [ ]:
x=np.array([[1.],[2.]]); y=np.array([[3.],[5.]])
w=np.zeros((1,1)); b=np.zeros(1)
loss,g=linear_loss(x,y,w,b)
assert loss==17 and g['w'].item()==-13 and g['b'].item()==-8
w-=.1*g['w']; b-=.1*g['b']
assert np.isclose(linear_loss(x,y,w,b)[0],1.685)
eps=1e-6
num=(linear_loss(x,y,w+eps,b)[0]-linear_loss(x,y,w-eps,b)[0])/(2*eps)
assert np.isclose(num,linear_loss(x,y,w,b)[1]['w'].item())
print(w,b,loss,num)

## Executed experiment
The measured experiment uses x in [-1,1], target 2x+.5, zero initialization and 60 full-batch steps. Only the learning rate changes: .001, .1, 1.1. The small rate progresses slowly; .1 converges; 1.1 destabilizes the intercept. For this centered problem the bias error multiplier is 1-2eta, whose magnitude exceeds one at eta>1. This bound is specific to this loss and scaling, not a universal neural-network learning-rate rule.

In [ ]:
print(record['optimizer'],record['seed'],record['dtype'])
print('Initial/final toy losses',record['toy_history'][0],record['toy_history'][-1])

## Practice
1. What gradient does f(w)=(w-3)² have at w=0? 2. What happens with eta=0? 3. Why must input scaling be fixed before comparing rates?

## Worked solutions
1. -6, so a positive learning rate moves w toward three. 2. Parameters never change. 3. Feature scale changes curvature and therefore the stable step sizes; a learning-rate comparison with different scaling is confounded.

In [ ]:
assert 2*(0-3)==-6
for row in record['linear']: print(row['learning_rate'],row['final_loss'])

## Evidence and interpretation
See [training curves](../../reports/mp13-mp15-training.png) and [complete record](../../reports/mp13-mp15-training.json). A decreasing training loss is an optimization observation, not proof of geographic generalization. The real red-to-NIR exercise has distinct scene splits and a training-mean baseline, but fixed numerical scaling is not physical calibration. Reflect on a case where the constant baseline could outperform the network.

## References
[PyTorch autograd tutorial](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html) and [optimization tutorial](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html). Models here are compact teaching examples, not paper reproductions or final SR systems.